# 逻辑回归第六课：ROC 与 AUC

这一课专门讲 ROC 曲线和 AUC，重点放在：

- 为什么阈值会影响分类结果
- TPR 和 FPR 分别是什么
- ROC 曲线是怎么画出来的
- AUC 代表什么含义
- 为什么 AUC 常被用来比较二分类模型


## 1. 为什么会需要 ROC 和 AUC

逻辑回归输出的是概率：

$$
p = P(y=1 \mid x)
$$

真正做分类时，我们通常再选一个阈值，例如 $0.5$：

- 如果 $p \ge 0.5$，预测为正类
- 如果 $p < 0.5$，预测为负类

问题在于：阈值一旦变化，混淆矩阵、precision、recall、accuracy 都会变化。

所以如果我们只看某一个固定阈值下的结果，就容易把模型本身的排序能力和阈值选择混在一起。

ROC 和 AUC 的作用，就是把“模型对正负样本的区分能力”单独拿出来看。

## 2. ROC 先关注两个量

ROC 曲线核心看两个量：

- 真正率 TPR
- 假正率 FPR

它们都来自混淆矩阵。

## 3. 什么是真正率 TPR

真正率也叫召回率 Recall，表示：

在所有真实正类中，被模型成功找出来的比例。

$$
TPR = \frac{TP}{TP+FN}
$$

如果 TPR 越大，说明模型漏掉的正类越少。

## 4. 什么是假正率 FPR

假正率表示：

在所有真实负类中，被模型错误判成正类的比例。

$$
FPR = \frac{FP}{FP+TN}
$$

如果 FPR 越大，说明模型误报的负类越多。

## 5. ROC 曲线是什么

ROC 曲线的横轴是 FPR，纵轴是 TPR：

$$
x\text{-axis} = FPR, \qquad y\text{-axis} = TPR
$$

当我们不断改变分类阈值时，每个阈值都会对应一组 $(FPR, TPR)$。

把这些点连起来，就得到 ROC 曲线。

因此 ROC 曲线不是某一个阈值下的结果，而是“所有阈值变化时的整体表现”。

## 6. 阈值变化时会发生什么

如果阈值很高，例如接近 $1$，模型会非常保守：

- 很少预测成正类
- TPR 往往较低
- FPR 往往也较低

如果阈值很低，例如接近 $0$，模型会非常激进：

- 很容易预测成正类
- TPR 会升高
- FPR 也会升高

所以 ROC 曲线其实是在看：

当我们尝试提高召回率时，要为此付出多少误报代价。

## 7. 一个很重要的参考线

ROC 图里常见一条对角线：

$$
TPR = FPR
$$

这条线代表“瞎猜”的水平。

原因是：如果模型完全没有区分能力，那么它把正类判成正类的比例，和把负类误判成正类的比例会差不多。

因此：

- ROC 曲线越靠近左上角越好
- 如果 ROC 曲线贴着对角线，说明模型区分能力很弱

## 8. AUC 是什么

AUC 的全称是：

$$
\mathrm{Area\ Under\ the\ ROC\ Curve}
$$

也就是 ROC 曲线下面积。

它把整条 ROC 曲线压缩成一个数字，范围通常在 $0$ 到 $1$ 之间。

一般来说：

- AUC 越大越好
- AUC = 0.5 附近，说明模型接近随机猜测
- AUC = 1，说明模型可以把正负样本完全排开

## 9. AUC 更直观的理解

AUC 还有一个非常常用的解释：

随机取一个正样本，再随机取一个负样本，模型给正样本的分数比负样本高的概率。

如果这个概率很高，说明模型排序能力强。

所以 AUC 本质上更关注的是：

模型能不能把正样本排在负样本前面。

这就是为什么 AUC 对阈值不敏感，而更关注整体排序质量。

## 10. 一个小例子

假设有 6 个样本，真实标签和模型分数分别是：

$$
y_{\text{true}} = [1, 1, 1, 0, 0, 0]
$$

$$
scores = [0.95, 0.80, 0.60, 0.55, 0.30, 0.10]
$$

如果阈值取 $0.5$，那么预测结果是：

$$
y_{\text{pred}} = [1, 1, 1, 1, 0, 0]
$$

这时：

- 三个正类都找出来了，所以 $TPR$ 很高
- 但有一个负类也被判成正类，所以 $FPR$ 不为 0

如果阈值继续提高，预测会更保守；如果阈值降低，预测会更激进。

ROC 就是把这种变化完整记录下来。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

y_true = np.array([1, 1, 1, 0, 0, 0])
y_score = np.array([0.95, 0.80, 0.60, 0.55, 0.30, 0.10])

fpr, tpr, thresholds = roc_curve(y_true, y_score)
auc = roc_auc_score(y_true, y_score)

print('fpr =', fpr)
print('tpr =', tpr)
print('thresholds =', thresholds)
print('auc =', auc)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, marker='o', label=f'ROC curve (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Example')
plt.legend()
plt.grid(True)
plt.show()


## 11. ROC 和 precision / recall 的区别

ROC 关注的是：

- 真正率 TPR
- 假正率 FPR

而 precision / recall 关注的是：

- 模型预测成正类之后，有多少是真的
- 所有真实正类中，有多少被找出来了

所以：

- ROC 更强调排序能力和整体区分能力
- precision / recall 更强调某个阈值下的实际分类效果

这两类指标并不冲突，而是看问题的角度不同。

## 12. 本节小结

这一节最核心的内容可以压缩成：

$$
TPR = \frac{TP}{TP+FN}
$$

$$
FPR = \frac{FP}{FP+TN}
$$

ROC 曲线是阈值变化时所有 $(FPR, TPR)$ 点连成的曲线。

AUC 是 ROC 曲线下面积，用来衡量模型整体排序能力。

如果要继续往下学，下一步最自然的内容就是：

- 把 ROC / AUC 接到你的逻辑回归脚本里
- 看阈值变化如何影响 precision 和 recall
- 学 PR 曲线